## Data acquisition and cleaning: PHMSA gas distribution incidents, 2010 - Present

In [5]:
import numpy as np 
import pandas as pd
from pathlib import Path
import re
import json

In [6]:
# Assign directories and ensure existence
DATA_DIR: Path = Path("..") / "data"
RAW_PATH: Path = DATA_DIR / "local" / "incident_gas_distribution_jan2010_present.txt" 
PROCESSED_DIR: Path = DATA_DIR / "local" / "processed"

assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR.resolve()}'
assert RAW_PATH.exists(), f'Raw file not found: {RAW_PATH.resolve()}'


## Load the raw data and keep it untouched

Upon load, every column is read as **text**, pandas is configured to not make **any** type guesses. We will handle every type explicitly to avoid any silent NaN conversions. raw_data is **never** modified, we will copy it to another variable and proceed with that variable from here.

In [7]:
EXPECTED_SHAPE: (int, int) = (1592, 468)

raw_data: pd.DataFrame = pd.read_csv(RAW_PATH, sep='\t', dtype=str, encoding='windows-1252', keep_default_na=False)

# Verify Shape
assert raw_data.shape == EXPECTED_SHAPE, "Shape incorrect!"
# Verify Unique Report Number
assert raw_data["REPORT_NUMBER"].is_unique, "Duplicate Report Numbers found!"
assert raw_data.columns.is_unique, "Duplicate column names found!"

# Preserve original raw load
data = raw_data.copy()

# Create log lists 
column_log: list[dict] = []
value_log: list[dict] = []
row_log: list[dict] = []

print(f'Loaded {raw_data.shape[0]:,} rows x {raw_data.shape[1]} columns')

Loaded 1,592 rows x 468 columns


## Schema Overview (read-only)

There are 438 columns — way too many to read with head(). The following cells will summarize each in different ways, **nothing here modifies data in any way**.

### Per-column Profile

In [8]:
PLACEHOLDER_TOKENS = ("UNKNOWN", "N/A", "NA", "NONE")


def profile_column(s: pd.Series, n_top: int = 4) -> dict:
    # Summarize one text column read with dtype=str, keep_default_na=False.
    values = s.str.strip()
    nonblank = values[values != ""]
    counts = nonblank.value_counts()
    real = nonblank[~nonblank.isin(PLACEHOLDER_TOKENS)]
    return {
        "column": s.name,
        "frac_blank": 1 - len(nonblank) / len(s),
        "n_nonblank": len(nonblank),
        "n_unique": len(counts),
        "frac_numeric": pd.to_numeric(nonblank, errors="coerce").notna().mean()
        if len(nonblank) else np.nan,
        "frac_numeric_excl_placeholders": pd.to_numeric(real, errors="coerce").notna().mean()
        if len(real) else np.nan,
        "top_values": " | ".join(f"{v} ({n})" for v, n in counts.head(n_top).items()),
    }


schema_profile = pd.DataFrame([profile_column(data[c]) for c in data.columns]).set_index("column")
assert len(schema_profile) == data.shape[1]

partly_filled = schema_profile[(schema_profile.frac_blank > 0) & (schema_profile.frac_blank < 1)]
partly_filled.sort_values("frac_blank").head(25)

,frac_blank,n_nonblank,n_unique,frac_numeric,frac_numeric_excl_placeholders,top_values
column,,,,,,
INCIDENT_AREA_SUBTYPE,0.001256,1590,14,0.000000,0.000000,TYPICAL ABOVEGROUND FACILITY PIPING OR APPURTE...
AUTHORIZER_EMAIL,0.001256,1590,554,0.000000,0.000000,rrsmith@nisource.com (37) | jason.pionk@cmsene...
EST_COST_EMERGENCY,0.003141,1587,777,1.000000,1.000000,0 (212) | 5000 (71) | 500 (69) | 1000 (53)
EST_COST_PROP_DAMAGE,0.003141,1587,843,1.000000,1.000000,0 (208) | 1000 (50) | 500 (41) | 5000 (41)
PREPARER_EMAIL,0.003141,1587,593,0.000000,0.000000,fagank@coned.com (45) | carmella.king@cmsenerg...
EST_COST_UNINTENTIONAL_RELEASE,0.003769,1586,868,1.000000,1.000000,0 (180) | 100 (37) | 10 (25) | 50 (25)
INCIDENT_IDENTIFIED_DATETIME,0.004397,1585,1584,0.000000,0.000000,6/23/2016 21:00 (2) | 7/28/2026 7:14 (1) | 7/1...
ON_SITE_DATETIME,0.005653,1583,1582,0.000000,0.000000,6/23/2016 21:00 (2) | 7/28/2026 7:35 (1) | 7/1...
NARRATIVE,0.006281,1582,1582,0.000000,0.000000,"ON JULY 28, 2026, A NATURAL GAS-FUELED EXPLOSI..."


### Column Groups 

Grouping by the name pattern/naming conventions of the column headers. Helpful to show the structure of the file in a few lines.

In [9]:

COLUMN_GROUP_PATTERNS = [
    ("personal_info", r"^(PREPARER|AUTHORIZER|LOCAL_CONTACT)_"),
    ("operator_address", r"^OPERATOR_(?!ID$)"),
    ("incident_address", r"^LOCATION_STREET_ADDRESS$"),
    ("admin_id", r"^(SUPPLEMENTAL_NUMBER|NRC_RPT_NUM|ADDITIONAL_NRC_REPORT_NUMBERS|ONE_CALL_TICKET_NUM)$"),
    ("cost", r"^(EST_COST_.*|PRPTY)$"),
    ("casualty", r"^(NUM_.*(FATALITIES|INJURIES)|FATAL|INJURE)$"),
    ("narrative_details_other", r"(^NARRATIVE$|_DETAILS$|_OTHER$)"),
    ("indicator", r"_IND$"),
]


def column_group(name: str) -> str:
    for group, pattern in COLUMN_GROUP_PATTERNS:
        if re.search(pattern, name):
            return group
    return "other"


schema_profile["group"] = [column_group(c) for c in schema_profile.index]

group_summary = (
    schema_profile.groupby("group")
    .agg(
        n_columns=("frac_blank", "size"),
        median_frac_blank=("frac_blank", "median"),
        n_fully_empty=("frac_blank", lambda x: int((x == 1).sum())),
    )
    .sort_values("n_columns", ascending=False)
)
group_summary


,n_columns,median_frac_blank,n_fully_empty
group,,,
indicator,190,0.997487,40
other,184,0.927136,12
narrative_details_other,51,0.994975,8
personal_info,12,0.023869,0
casualty,12,0.788945,0
cost,8,0.005339,0
operator_address,6,0.000000,0
admin_id,4,0.375000,0
incident_address,1,0.000000,0


In [10]:
for group in ["personal_info", "operator_address", "incident_address", "admin_id"]:
    members = schema_profile.index[schema_profile.group == group].tolist()
    print(f"{group} ({len(members)}): {members}")

personal_info (12): ['PREPARER_NAME', 'PREPARER_TITLE', 'PREPARER_EMAIL', 'PREPARER_TELEPHONE', 'PREPARER_FAX', 'LOCAL_CONTACT_NAME', 'LOCAL_CONTACT_EMAIL', 'LOCAL_CONTACT_TELEPHONE', 'AUTHORIZER_NAME', 'AUTHORIZER_TITLE', 'AUTHORIZER_TELEPHONE', 'AUTHORIZER_EMAIL']
operator_address (6): ['OPERATOR_STREET_ADDRESS', 'OPERATOR_CITY_NAME', 'OPERATOR_STATE_ABBREVIATION', 'OPERATOR_POSTAL_CODE', 'OPERATOR_TYPE', 'OPERATOR_QUALIFICATION_IND']
incident_address (1): ['LOCATION_STREET_ADDRESS']
admin_id (4): ['SUPPLEMENTAL_NUMBER', 'NRC_RPT_NUM', 'ADDITIONAL_NRC_REPORT_NUMBERS', 'ONE_CALL_TICKET_NUM']


### Entirely Empty Columns

In [11]:
entirely_empty_columns = schema_profile.index[schema_profile.frac_blank == 1].tolist()
print(f"{len(entirely_empty_columns)} columns are 100% blank.")

60 columns are 100% blank.


### Report-form versions

PHMSA updated the incident report form during this period (2010 - present), so some columns that exist on only one version, the previous one, are blank for every report filed on the other. This doesn't classify as missing data, it just means that that question wasn't previously asked. We use the TIME_ZONE marker instead of the incident date because a report's form depends on when it was filed, not when the incident happened. 

In [12]:
NEW_FORM_MARKER = "TIME_ZONE"

nonblank = data.apply(lambda s: s.str.strip() != "")
on_new_form = nonblank[NEW_FORM_MARKER]

received = pd.to_datetime(data["REPORT_RECEIVED_DATE"].str.strip(), format="%m/%d/%Y", errors="coerce")
print(f"Reports on the new form: {on_new_form.sum():,} of {len(data):,} ({on_new_form.mean():.1%})")
print(f"Received-date range, old form: {received[~on_new_form].min():%Y-%m-%d} to {received[~on_new_form].max():%Y-%m-%d}")
print(f"Received-date range, new form: {received[on_new_form].min():%Y-%m-%d} to {received[on_new_form].max():%Y-%m-%d}")

form_fill = pd.DataFrame({
    "fills_old_form": nonblank[~on_new_form].sum(),
    "fills_new_form": nonblank[on_new_form].sum(),
})
pd.crosstab(data["IYEAR"], on_new_form.rename("on_new_form"))

Reports on the new form: 450 of 1,592 (28.3%)
Received-date range, old form: 2010-02-19 to 2020-12-04
Received-date range, new form: 2016-09-09 to 2026-08-26


on_new_form,False,True
IYEAR,,
2010,120,0
2011,116,0
2012,88,0
2013,104,0
2014,106,0
2015,101,0
2016,113,2
2017,103,0
2018,102,6


In [13]:
# A column is form-version-only if it is never filled on one form but filled on the other
# often enough (>= MIN_FILLS) that this is a pattern, not a coincidence.
MIN_FILLS = 10

new_form_only = form_fill.index[(form_fill.fills_old_form == 0) & (form_fill.fills_new_form >= MIN_FILLS)]
old_form_only = form_fill.index[(form_fill.fills_new_form == 0) & (form_fill.fills_old_form >= MIN_FILLS)]
form_version_only = sorted(set(new_form_only) | set(old_form_only))

# Cost and casualty components are exempt: PRPTY and FATAL are sums over all of them, so
# dropping one (e.g. a cost line added on the new form) would break those totals.
exempt_groups = {"cost", "casualty"}
form_version_exempt = [c for c in form_version_only if schema_profile.at[c, "group"] in exempt_groups]
form_version_only = [c for c in form_version_only if c not in form_version_exempt]

print(f"New-form-only columns ({len(new_form_only)}): {list(new_form_only)}")
print(f"Old-form-only columns ({len(old_form_only)}): {list(old_form_only)}")
print(f"Kept despite form version (cost/casualty components): {form_version_exempt}")

New-form-only columns (59): ['TIME_ZONE', 'DAYLIGHT_SAVINGS_IND', 'FLOW_CONT_KEY_CRIT_IND', 'FLOW_CONT_MAIN_VALVE_IND', 'FLOW_CONT_SERVICE_VALVE_IND', 'FLOW_CONT_METER_REG_IND', 'FLOW_CONT_SQUEEZE_OFF_IND', 'FLOW_CONT_STOPPLE_FITNG_IND', 'FLOW_CONT_OTHER_IND', 'FLOW_CONT_OTHER_DETAIL', 'ADDITIONAL_NRC_REPORT_NUMBERS', 'INTENTIONAL_RELEASE', 'IGNITE_DATETIME', 'HOW_EXTINGUISHED', 'HOW_EXTINGUISHED_OTHER_DETAIL', 'GAS_CONSUMED_BY_FIRE_IN_MCF', 'COMMUNICATION_STATE_FED_IND', 'PARTY_INITIATED_COMMUNICATION', 'INITIAL_RESPONDER_COM_DATETIME', 'CONFIRMED_DISCOVERY_DATETIME', 'OTHER_UNDERGROUND_FACILITIES', 'MANUFACTURED_YEAR', 'CUSTOMER_TYPE', 'WAS_EFV_INSTALLED_BEFORE_IND', 'EVF_ACTIVATION_IND', 'CURB_VALVE_INST_BEFORE_INC_IND', 'GAS_COST_IN_MCF', 'EST_COST_INTENTIONAL_RELEASE', 'NUM_PERSONS_HOSP_NOT_OVNGHT', 'NUM_INJURED_TREATED_BY_EMT', 'NUM_RESIDENT_BUILDING_AFFCTD', 'NUM_BUSINESS_BUILDING_AFFCTD', 'MOP_CFR_SECTION', 'MAOP_ESTABLISHED_DATE', 'GAS_ODORIZED_SYSTEM_TYPE', 'GAS_ODORIZED_SYS_

### Placeholder tokens

"Not Known" values must be converted to NaN before numeric conversion. In a categorical column, NONE is valid.

In [14]:
placeholder_counts = pd.DataFrame(
    {token: data.apply(lambda s, t=token: (s.str.strip() == t).sum()) for token in PLACEHOLDER_TOKENS}
)
placeholder_counts = placeholder_counts[placeholder_counts.sum(axis=1) > 0]
placeholder_counts.join(schema_profile[["frac_numeric_excl_placeholders", "top_values"]])

,UNKNOWN,N/A,NA,NONE,frac_numeric_excl_placeholders,top_values
LOCATION_STREET_ADDRESS,3,0,0,0,0.000000,UNKNOWN (3) | ENTIRE CITY (2) | VAR. LOC'S IN ...
LOCATION_COUNTY_NAME,0,1,1,0,0.000000,LOS ANGELES (34) | DALLAS (24) | WAYNE (23) | ...
FLOW_CONT_OTHER_DETAIL,0,1,0,1,0.000000,BRASS PLUG ON CAST IRON MAIN (1) | SERVICE LIN...
ADDITIONAL_NRC_REPORT_NUMBERS,0,3,2,0,0.921671,N/A (3) | NA (2) | 1427203 (2) | 1469420 (1)
INCIDENT_AREA_DETAILS,2,0,0,0,0.000000,BASEMENT (2) | NTSB INVESTIGATION ONGOING (2) ...
WATER_NAME,1,0,0,0,0.000000,COTTONWOOD RIVER (2) | BAYOU DINDE (1) | THORN...
WATER_DEPTH,4,0,0,0,1.000000,UNKNOWN (4) | 1 (3) | 13 (1) | 27 (1)
SYSTEM_PART_DETAILS,8,0,0,0,0.000000,"UNKNOWN (8) | ONSHORE PIPELINE, INCLUDING VALV..."
INSTALLATION_YEAR,280,0,0,0,1.000000,UNKNOWN (280) | 1969 (29) | 1994 (28) | 2002 (28)
MANUFACTURED_YEAR,334,0,0,0,1.000000,UNKNOWN (334) | 2020 (6) | 2021 (5) | 1992 (4)


### Indicator Shapes

- Yes/blank only: checkbox. Blank means not checked, so we encode it as 0, not as a missing value. 
- Yes **and** No: a real yes/no category. Blank or UNKNOWN means not answered, so remains as missing. 

In [15]:
def value_set(s: pd.Series) -> frozenset:
    stripped = s.str.strip()
    return frozenset(stripped[stripped != ""].unique())


value_sets = {c: value_set(data[c]) for c in data.columns}
yes_only_cols = [c for c, v in value_sets.items() if v == {"YES"}]
# YES/NO questions may also carry a placeholder (e.g. UNKNOWN), which becomes missing.
yes_no_cols = [
    c for c, v in value_sets.items()
    if {"YES", "NO"} <= v and v <= {"YES", "NO", *PLACEHOLDER_TOKENS}
]
other_ind_cols = [
    c for c in data.columns
    if c.endswith("_IND") and c not in yes_only_cols and c not in yes_no_cols and value_sets[c]
]

print(f"YES/blank checkbox columns: {len(yes_only_cols)}")
print(f"YES/NO question columns:   {len(yes_no_cols)}")
print("*_IND columns with other value sets (review):", {c: sorted(value_sets[c]) for c in other_ind_cols})

YES/blank checkbox columns: 114
YES/NO question columns:   38
*_IND columns with other value sets (review): {'EVF_ACTIVATION_IND': ['NO', 'UNABLE TO DETERMINE', 'YES'], 'WT_PLASTIC_UNKNOWN_IND': ['UNKNOWN'], 'STATE_LAW_EXEMPT_IND': ['NO', 'UNKNOWN'], 'WHITE_LINED_IND': ['NO', 'Unknown', 'YES'], 'OSF_HYDROTEST_CONDUCTED_IND': ['NO'], 'OTHER_STRIPPED_IND': ['OTHER', 'PIPE NIPPLE', 'THREADED FITTING'], 'OTHER_NON_THREADED_IND': ['GASKET', 'OTHER', 'OTHER SEAL OR PACKING'], 'VALVE_OTHER_IND': ['OTHER']}


## Convert blanks and placeholders -> NaN

In [16]:
data = data.apply(lambda s: s.str.strip())
data = data.mask(data == "")

data["installation_year_unknown"] = (data["INSTALLATION_YEAR"] == "UNKNOWN").astype("Int8")
print(f"installation_year_unknown: {int(data['installation_year_unknown'].sum())} reports")

ID_COLUMN_PATTERN = r"(^REPORT_NUMBER$|^SUPPLEMENTAL_NUMBER$|^OPERATOR_ID$|_POSTAL_CODE$|TELEPHONE|_FAX$|NRC|ONE_CALL_TICKET)"
DATE_COLUMN_PATTERN = r"(DATE|DATETIME)$"

id_columns = [c for c in data.columns if re.search(ID_COLUMN_PATTERN, c)]
date_columns = [c for c in data.columns if re.search(DATE_COLUMN_PATTERN, c)]


def numeric_parse_rate(s: pd.Series) -> tuple[int, float]:
    real = s.dropna()
    real = real[~real.isin(PLACEHOLDER_TOKENS)]
    if real.empty:
        return 0, np.nan
    return len(real), pd.to_numeric(real, errors="coerce").notna().mean()


THOUSANDS_PATTERN = r"^-?\d{1,3}(,\d{3})+(\.\d+)?$"
text_columns = [
    c for c in data.columns
    if c not in id_columns and c not in date_columns and c != "installation_year_unknown"
]

# Thousands separators: strip commas only where every non-numeric value is a well-formed "1,234".
for column in text_columns:
    real = data[column].dropna()
    real = real[~real.isin(PLACEHOLDER_TOKENS)]
    non_numeric = real[pd.to_numeric(real, errors="coerce").isna()]
    if len(non_numeric) and non_numeric.str.match(THOUSANDS_PATTERN).all():
        has_comma = data[column].str.match(THOUSANDS_PATTERN, na=False)
        value_log.append({
            "step": "4 thousands separators",
            "column": column,
            "n_values": int(has_comma.sum()),
            "rule": "removed thousands separators",
        })
        data[column] = data[column].str.replace(",", "", regex=False)

parse_rates = {c: numeric_parse_rate(data[c]) for c in text_columns}
numeric_candidates = [c for c, (n, rate) in parse_rates.items() if n > 0 and rate == 1.0]
partly_numeric = {c: rate for c, (n, rate) in parse_rates.items() if n > 0 and 0.5 <= rate < 1.0}

for column in numeric_candidates + date_columns:
    is_placeholder = data[column].isin(PLACEHOLDER_TOKENS)
    if is_placeholder.any():
        value_log.append({
            "step": "4 placeholders",
            "column": column,
            "n_values": int(is_placeholder.sum()),
            "rule": "placeholder token -> NaN",
        })
        data[column] = data[column].mask(is_placeholder)

step4 = [r for r in value_log if r["step"].startswith("4")]
print(f"Numeric candidates: {len(numeric_candidates)} columns")
print(f"Step 4 changed {sum(r['n_values'] for r in step4):,} values in {len(step4)} columns")
pd.DataFrame(step4)


installation_year_unknown: 280 reports
Numeric candidates: 68 columns
Step 4 changed 1,102 values in 6 columns


,step,column,n_values,rule
0,4 placeholders,WATER_DEPTH,4,placeholder token -> NaN
1,4 placeholders,INSTALLATION_YEAR,280,placeholder token -> NaN
2,4 placeholders,MANUFACTURED_YEAR,334,placeholder token -> NaN
3,4 placeholders,WT_STEEL,366,placeholder token -> NaN
4,4 placeholders,MATERIAL_PE_PIPE_CODE,79,placeholder token -> NaN
5,4 placeholders,ESTIMATED_SPEED,39,placeholder token -> NaN


In [17]:
# Columns that are mostly but not entirely numeric: review the offending values.
for column, rate in sorted(partly_numeric.items(), key=lambda kv: -kv[1]):
    values = data[column].dropna()
    offending = values[pd.to_numeric(values, errors="coerce").isna()].value_counts().head(5)
    print(f"{column}: {rate:.1%} numeric; non-numeric examples: {offending.to_dict()}")

if not partly_numeric:
    print("No partly-numeric columns: every column is either cleanly numeric or text.")

PLASTIC_SDR: 98.5% numeric; non-numeric examples: {'DR11': 2, '4"': 1, '5/8"': 1, '8-May': 1}


## Drop columns, but for domain reasons **only**

All other column drops will be related to different functions like different regressions and analysis. 

| Reason | Why |
|---|---|
| `personal_info` | Names, emails and phone numbers of preparers and contacts. Not predictive, and this repository is public. |
| `operator_address`, `incident_address` | Street-level location. Coarse location (state, ZIP, lat/lon) is kept. |
| `admin_id` | Filing identifiers. `ONE_CALL_TICKET_NUM` was also corrupted into scientific notation upstream. |
| `free_text` | The narrative and high-cardinality "details/other, specify" text. This project does no text modeling. Low-cardinality `*_DETAILS` columns (e.g. sub-cause categories) are structured and are **kept**. |
| `fully_empty` | No information. |
| `form_version_only` | Exists on only one form version (step 3d), so it is undefined for the other version's reports. Cost and casualty components are exempt, because `PRPTY` and `FATAL` are totals over all of them. |


In [18]:
FREE_TEXT_MIN_UNIQUE = 50  # "details/other" columns with more distinct values than this are free text

details_other = [c for c in data.columns if re.search(r"_DETAILS$|_OTHER$", c) and c not in numeric_candidates]
free_text = ["NARRATIVE"] + [c for c in details_other if data[c].nunique() > FREE_TEXT_MIN_UNIQUE]
kept_details_other = sorted(set(details_other) - set(free_text))

drop_reasons = {
    "personal_info": schema_profile.index[schema_profile.group == "personal_info"].tolist(),
    "operator_address": schema_profile.index[schema_profile.group == "operator_address"].tolist(),
    "incident_address": ["LOCATION_STREET_ADDRESS"],
    "admin_id": ["SUPPLEMENTAL_NUMBER", "NRC_RPT_NUM", "ADDITIONAL_NRC_REPORT_NUMBERS", "ONE_CALL_TICKET_NUM"],
    "free_text": free_text,
    "fully_empty": [c for c in data.columns if data[c].isna().all() and c != "installation_year_unknown"],
    "form_version_only": form_version_only,
}

assert drop_reasons["personal_info"], "Expected personal-information columns; check the regex."

# Every column gets the first reason that applies, so the counts add up.
assigned: dict[str, str] = {}
for reason, columns in drop_reasons.items():
    missing = [c for c in columns if c not in data.columns]
    assert not missing, f"{reason}: columns not in data: {missing}"
    for column in columns:
        assigned.setdefault(column, reason)

PROTECTED = {"REPORT_NUMBER", "OPERATOR_ID", "LOCAL_DATETIME", "IYEAR", "REPORT_TYPE", "CAUSE", "PRPTY"}
assert not PROTECTED & assigned.keys(), f"Refusing to drop key columns: {PROTECTED & assigned.keys()}"

column_log = [{"column": c, "reason": r} for c, r in assigned.items()]
data = data.drop(columns=list(assigned))

print(f"Dropped {len(assigned)} columns; {data.shape[1]} remain.")
print(f"Kept structured *_DETAILS/*_OTHER columns: {kept_details_other}")
pd.Series(assigned).value_counts().rename("n_dropped").to_frame()

Dropped 151 columns; 318 remain.
Kept structured *_DETAILS/*_OTHER columns: ['ADDITIONAL_FACTOR_DETAILS', 'BUTT_WELD_DETAILS', 'CAUSE_DETAILS', 'COATING_TYPE_DETAILS', 'COMMODITY_DETAILS', 'CORROSION_BASIS_DETAILS', 'CORROSION_LOCATION_DETAILS', 'CORROSION_TYPE_DETAILS', 'EQ_FAILURE_DETAILS', 'FILLET_WELD_DETAILS', 'FPW_FIRST_PLASTIC_TYPE_OTHER', 'FPW_SECOND_PLASTIC_TYPE_OTHER', 'INTENTIONAL_DETAILS', 'INT_CORROSION_BASIS_DETAILS', 'INT_CORROSION_TYPE_DETAILS', 'INT_VISUAL_EXAM_DETAILS', 'INVEST_NO_SCHEDULE_IND_DETAILS', 'INVEST_OTHER_IND_DETAILS', 'NF_EXTREME_WEATHER_DETAILS', 'NF_OTHER_DETAILS', 'OPERATION_RELATED_DETAILS', 'OSF_OTHER_WEATHER_DETAILS', 'OTHER_CONTROL_RELIEF_DETAILS', 'OTHER_CP_SURVEY_DETAILS', 'OTHER_NON_THREADED_DETAILS', 'OTHER_STRIPPED_DETAILS', 'PIPE_BODY_DETAILS', 'PIPE_SEAM_DETAILS', 'PIPE_TYPE_OTHER', 'PLASTIC_DETAILS', 'PLASTIC_JOINT_DETAILS', 'PWJF_FAILURE_DETAILS', 'RESULT_MATERIAL_DETAILS', 'ROOT_CAUSE_TYPE_OTHER', 'RUPTURE_DETAILS', 'STRAY_CURRENT_DETAILS

,n_dropped
fully_empty,60
form_version_only,55
free_text,13
personal_info,12
operator_address,6
admin_id,4
incident_address,1


## Fix all types

- Dates are parsed in format M/D/YYYY (with or without a time). Values that do not fit this will raise an error. 
- Numbers are numerically converted
- Indicators: Yes/Blank become 0/1 (blank means not checked). Yes/No questions become nullable booleans (True/False/Blank), with blank staying unknown
- ZIP Codes are kept as text, minimized to 5 digits. Values that are **not** ZIP become NaN
- IDs remain as text

### Dates

In [19]:
DATE_FORMATS = ("%m/%d/%Y %H:%M", "%m/%d/%Y %H:%M:%S", "%m/%d/%Y")


def parse_dates_strict(s: pd.Series) -> pd.Series:
    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in DATE_FORMATS:
        todo = parsed.isna() & s.notna()
        if not todo.any():
            break
        parsed[todo] = pd.to_datetime(s[todo], format=fmt, errors="coerce")
    failed = s.notna() & parsed.isna()
    if failed.any():
        raise ValueError(f"{s.name}: {failed.sum()} unparseable dates, e.g. {s[failed].unique()[:5]}")
    return parsed


date_columns = [c for c in data.columns if re.search(DATE_COLUMN_PATTERN, c)]
for column in date_columns:
    data[column] = parse_dates_strict(data[column])

assert data["LOCAL_DATETIME"].notna().all(), "Every report needs an incident timestamp."
print(f"Parsed {len(date_columns)} date columns: {date_columns}")

Parsed 8 date columns: ['REPORT_RECEIVED_DATE', 'LOCAL_DATETIME', 'NRC_RPT_DATETIME', 'INCIDENT_IDENTIFIED_DATETIME', 'ON_SITE_DATETIME', 'COR_HYDROTEST_LEAK_SURVEY_DATE', 'EX_HYDROTEST_LEAK_SURVEY_DATE', 'OSF_HYDROTEST_LEAK_SURVEY_DATE']


### Numeric Columns

In [20]:
numeric_columns = [c for c in numeric_candidates if c in data.columns]
nan_before = data[numeric_columns].isna().sum()

data[numeric_columns] = data[numeric_columns].apply(pd.to_numeric)

new_nans = (data[numeric_columns].isna().sum() - nan_before)
assert (new_nans == 0).all(), f"Numeric conversion created NaN: {new_nans[new_nans > 0].to_dict()}"

data["IYEAR"] = data["IYEAR"].astype("Int64")
print(f"Converted {len(numeric_columns)} numeric columns.")

Converted 56 numeric columns.


### Checkboxes and Yes/No Columns

In [21]:
yes_only = [c for c in yes_only_cols if c in data.columns]
yes_no = [c for c in yes_no_cols if c in data.columns]

for column in yes_only:
    data[column] = (data[column] == "YES").astype("Int8")
for column in yes_no:
    # Anything other than YES/NO (blank, UNKNOWN) maps to missing.
    data[column] = data[column].map({"YES": True, "NO": False}).astype("boolean")

print(f"Encoded {len(yes_only)} checkbox columns (0/1) and {len(yes_no)} yes/no columns (nullable bool).")

Encoded 103 checkbox columns (0/1) and 26 yes/no columns (nullable bool).


### Zip Codes

In [22]:
zip_raw = data["LOCATION_POSTAL_CODE"]
zip5 = zip_raw.str.extract(r"^(\d{5})(?:-?\d{4})?$", expand=False)
invalid_zip = zip_raw.notna() & zip5.isna()

value_log.append({
    "step": "6 types",
    "column": "LOCATION_POSTAL_CODE",
    "n_values": int(invalid_zip.sum()),
    "rule": "not a 5-digit ZIP or ZIP+4 -> NaN",
})
print(f"ZIP+4 trimmed to 5 digits: {int((zip5.notna() & (zip_raw.str.len() > 5)).sum())}")
print(f"Invalid ZIPs set to NaN: {zip_raw[invalid_zip].value_counts().to_dict()}")
data["LOCATION_POSTAL_CODE"] = zip5

# Every remaining text column (IDs, categories, ZIPs) gets pandas' dedicated string dtype, so
# missing text is pd.NA everywhere and the Parquet round trip is exact on pandas 2 and 3.
text_columns = [
    c for c in data.columns
    if pd.api.types.is_object_dtype(data[c].dtype) or isinstance(data[c].dtype, pd.StringDtype)
]
data[text_columns] = data[text_columns].astype("string")
assert data["REPORT_NUMBER"].dtype == "string" and data["OPERATOR_ID"].dtype == "string"

data.dtypes.astype(str).value_counts().rename("n_columns").to_frame()

ZIP+4 trimmed to 5 digits: 186
Invalid ZIPs set to NaN: {'81501-': 1}


,n_columns
string,124
Int8,104
float64,49
boolean,26
datetime64[ns],8
int64,6
Int64,1


## Convert physically impossible values -> NaN

Generally speaking, a bad value in one field does not invalidate the incident. Instead, we null that value and **keep** the row. Dropping the row would remove a valid label and tons of valid features. 

In [23]:
print("INSTALLATION_YEAR counts, 1895-1910 (a spike at 1900 suggests a placeholder):")
print(data["INSTALLATION_YEAR"].loc[data["INSTALLATION_YEAR"].between(1895, 1910)].value_counts().sort_index().to_dict())

print("\nLargest DEPTH_OF_COVER values (inches):")
print(data["DEPTH_OF_COVER"].nlargest(8).tolist())

print("\nZero pressures:")
for column in ["ACCIDENT_PSIG", "NORMAL_PSIG", "MOP_PSIG"]:
    if column in data.columns:
        print(f"  {column} == 0: {int((data[column] == 0).sum())}")

print(f"\nPIPE_DIAMETER <= 0: {int((data['PIPE_DIAMETER'] <= 0).sum())}")
print(f"Longitude > 0 (east of Greenwich): {int((data['LOCATION_LONGITUDE'] > 0).sum())}")

INSTALLATION_YEAR counts, 1895-1910 (a spike at 1900 suggests a placeholder):
{1900.0: 3, 1901.0: 2, 1902.0: 1, 1903.0: 2, 1904.0: 1, 1906.0: 1, 1907.0: 2, 1909.0: 1}

Largest DEPTH_OF_COVER values (inches):
[840.0, 360.0, 358.0, 348.0, 300.0, 300.0, 300.0, 300.0]

Zero pressures:
  ACCIDENT_PSIG == 0: 16
  NORMAL_PSIG == 0: 9
  MOP_PSIG == 0: 8

PIPE_DIAMETER <= 0: 0
Longitude > 0 (east of Greenwich): 0


**Decisions:**

| Value | Decision | Reason |
|---|---|---|
| `PIPE_DIAMETER <= 0` | NaN | A pipe cannot have zero diameter. |
| `INSTALLATION_YEAR` after the incident year | NaN | A pipe cannot fail before it is installed. |
| `INSTALLATION_YEAR == 1900` | NaN and `installation_year_unknown = 1`, **if** the counts above show a spike at 1900 | A spike at a round year is a placeholder, not a real installation date. Set `TREAT_1900_AS_PLACEHOLDER` accordingly. |
| Negative costs or casualty counts | NaN | Impossible by definition. |
| Latitude outside ±90, longitude outside ±180 | NaN | Not a location on Earth. |
| Large `DEPTH_OF_COVER` (e.g. 840 in = 70 ft) | **Kept** | Horizontally drilled crossings under rivers and highways can be this deep. Revisit if the field-description file says otherwise. |
| Zero pressures | **Kept** | Low-pressure distribution systems run at a few inches of water column (≈0.25 psig), which can legitimately be recorded as 0. |
| Accident pressure above MOP | **Kept** | Overpressure really happens and is a failure mechanism, not a recording error. |

In [24]:
TREAT_1900_AS_PLACEHOLDER = True  # set after reading the counts above

incident_year = data["LOCAL_DATETIME"].dt.year
nonnegative_columns = [
    c for c in data.columns
    if re.search(r"^(EST_COST_.*|PRPTY|NUM_.*|FATAL|INJURE)$", c) and pd.api.types.is_numeric_dtype(data[c])
]

impossible_rules = [
    ("PIPE_DIAMETER", data["PIPE_DIAMETER"] <= 0, "diameter <= 0"),
    ("INSTALLATION_YEAR", data["INSTALLATION_YEAR"] > incident_year, "installed after the incident year"),
    ("LOCATION_LATITUDE", ~data["LOCATION_LATITUDE"].between(-90, 90) & data["LOCATION_LATITUDE"].notna(), "latitude outside [-90, 90]"),
    ("LOCATION_LONGITUDE", ~data["LOCATION_LONGITUDE"].between(-180, 180) & data["LOCATION_LONGITUDE"].notna(), "longitude outside [-180, 180]"),
]
if TREAT_1900_AS_PLACEHOLDER:
    is_1900 = data["INSTALLATION_YEAR"] == 1900
    impossible_rules.append(("INSTALLATION_YEAR", is_1900, "1900 used as a placeholder year"))
    # A placeholder year means the operator did not know the age, the same as "UNKNOWN".
    data.loc[is_1900, "installation_year_unknown"] = 1
impossible_rules += [(c, data[c] < 0, "negative cost or count") for c in nonnegative_columns]

# All masks were computed on the same snapshot above, so the order of application doesn't matter.
for column, mask, rule in impossible_rules:
    mask = mask.fillna(False).astype(bool)
    value_log.append({"step": "7 impossible", "column": column, "n_values": int(mask.sum()), "rule": rule})
    if mask.any():
        data[column] = data[column].mask(mask)  # upcasts int -> float when needed

pd.DataFrame([r for r in value_log if r["step"] == "7 impossible" and r["n_values"] > 0])

,step,column,n_values,rule
0,7 impossible,INSTALLATION_YEAR,3,1900 used as a placeholder year


## Rows: duplicates, report status, and removal log

Rows are only removed if they are not valid observations. Currently, this removes nothing. 

- **Exact** duplicates are removed, and counted. 
- Some reports share the same timestamp but are in different locations, hence they are different incidents and both are kept. 
- Non-Final reports are kept with an is_final tag. Their cause is rarely revised, so the classification notebook will keep them. These decisions depend on the individual task notebooks. 
- 2026 is a partial year, but the reports are valid incidents so they are kept. 

In [25]:
rows_before = len(data)
data = data.drop_duplicates()
row_log.append({
    "step": "8 exact duplicates",
    "rows_before": rows_before,
    "rows_after": len(data),
    "removed": rows_before - len(data),
    "reason": "identical in every column",
})

assert data["REPORT_NUMBER"].is_unique

same_time_place = data.duplicated(subset=["LOCAL_DATETIME", "LOCATION_LATITUDE", "LOCATION_LONGITUDE"], keep=False)
print(f"Reports sharing timestamp and coordinates: {int(same_time_place.sum())}")

shared_timestamp = data[data.duplicated(subset=["LOCAL_DATETIME"], keep=False)]
display(shared_timestamp[["REPORT_NUMBER", "LOCAL_DATETIME", "OPERATOR_ID", "LOCATION_CITY_NAME", "REPORT_TYPE"]])

KNOWN_DISTINCT_PAIR = {"20160110", "20160111"}
assert KNOWN_DISTINCT_PAIR <= set(data["REPORT_NUMBER"]), "WV flood pair should both be kept."

Reports sharing timestamp and coordinates: 0


,REPORT_NUMBER,LOCAL_DATETIME,OPERATOR_ID,LOCATION_CITY_NAME,REPORT_TYPE
907,20160110,2016-06-23 21:00:00,12878,WHITE SULPHUR SPRINGS,ORIGINAL FINAL
908,20160111,2016-06-23 21:00:00,12878,ELKVIEW,ORIGINAL FINAL


In [26]:
assert data["REPORT_TYPE"].notna().all(), "Every report needs a REPORT_TYPE."
report_types = set(data["REPORT_TYPE"].unique())
assert all(t.startswith(("ORIGINAL", "SUPPLEMENTAL")) for t in report_types), report_types

data["is_final"] = data["REPORT_TYPE"].str.contains("FINAL", regex=False).astype(bool)

print(data["REPORT_TYPE"].value_counts().to_dict())
pd.crosstab(data["IYEAR"], data["is_final"]).tail(6)

{'SUPPLEMENTAL FINAL': 974, 'ORIGINAL FINAL': 438, 'ORIGINAL': 127, 'SUPPLEMENTAL': 53}


is_final,False,True
IYEAR,,
2021,3,84
2022,5,55
2023,5,66
2024,13,49
2025,18,43
2026,27,18


In [27]:
row_summary = pd.DataFrame(row_log)
print(f"Rows in: {len(raw_data):,}   Rows out: {len(data):,}")
row_summary

Rows in: 1,592   Rows out: 1,592


,step,rows_before,rows_after,removed,reason
0,8 exact duplicates,1592,1592,0,identical in every column
